# Notebook 02 — Estructuras de control de flujo

Segundo sub-bloque del Tema 06. Con bloques anónimos y variables del Notebook 01, ahora agregamos **lógica de decisión** (`IF`, `CASE`) y **lógica de iteración** (`LOOP`, `FOR`, `WHILE`). Esto es lo que convierte PL/pgSQL en un lenguaje procedural completo.

Vas a ver tres patrones de `FOR` que es importante distinguir: sobre **rangos numéricos**, sobre **filas de queries**, y sobre **elementos de arrays**. Cada uno se ve igual pero hace cosas distintas.

**Contenido de este notebook:**

- [Setup](#setup)
- [`IF` con sus variantes](#if-con-sus-variantes)
- [`CASE` — expresión vs sentencia](#case--expresión-vs-sentencia)
- [`FOR` sobre rangos numéricos](#for-sobre-rangos-numéricos)
- [`FOR` sobre filas de query](#for-sobre-filas-de-query)
- [`WHILE`](#while)

## Setup

In [ ]:
# Setup — instala JupySQL si hace falta (Colab trae ipython-sql, no JupySQL).
import importlib.util, subprocess, sys
if importlib.util.find_spec("jupysql") is None:
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "ipython-sql"], check=False)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "jupysql"], check=True)
    print("⚠ JupySQL instalado. REINICIA el kernel (Entorno de ejecución → Reiniciar sesión)")
    print("  y vuelve a correr esta celda y las siguientes.")
else:
    print("✓ JupySQL listo.")

In [ ]:
%load_ext sql

from sqlalchemy import create_engine

# Reemplaza con tus valores del Tema 01
AURORA_HOST     = "aurora-mod4.cluster-xxxxx.us-east-1.rds.amazonaws.com"
AURORA_PASSWORD = "TU_PASSWORD_AQUI"
AURORA_DATABASE = "northwind"

engine = create_engine(
    f"postgresql+psycopg2://postgres:{AURORA_PASSWORD}@{AURORA_HOST}:5432/{AURORA_DATABASE}"
)

%sql engine
# Devuelve cada query como DataFrame de pandas (mejor render en Colab, y
# el resultado es directamente manipulable con pandas).
%config SqlMagic.autopandas = True

## `IF` con sus variantes

Las tres formas de `IF` en PL/pgSQL, de más simple a más completa:

```sql
-- 1. IF simple
IF condicion THEN
    ...
END IF;

-- 2. IF con ELSE
IF condicion THEN
    ...
ELSE
    ...
END IF;

-- 3. IF encadenado con ELSIF
IF cond1 THEN
    ...
ELSIF cond2 THEN
    ...
ELSIF cond3 THEN
    ...
ELSE
    ...
END IF;
```

**Detalles:** `ELSIF` (sin `E` final, distinto a otros lenguajes), termina con `END IF;` (dos palabras, no `ENDIF`), las condiciones son booleanas SQL (lo que pondrías en un `WHERE`).

In [ ]:
%%sql
DO $$
DECLARE
    total_clientes INTEGER;
    categoria      TEXT;
BEGIN
    SELECT COUNT(*) INTO total_clientes FROM northwind_dwh.dim_customer;
    
    IF total_clientes > 500 THEN
        categoria := 'enterprise';
    ELSIF total_clientes > 100 THEN
        categoria := 'mediana';
    ELSIF total_clientes > 10 THEN
        categoria := 'pequeña';
    ELSE
        categoria := 'micro';
    END IF;
    
    RAISE NOTICE 'Total clientes: %, categoría: %', total_clientes, categoria;
END
$$;

## `CASE` — expresión vs sentencia

PL/pgSQL hereda el `CASE` de SQL, pero agrega una **variante sentencia** que puede ejecutar bloques de código:

### Variante 1 — `CASE` como expresión

La que ya conoces de SQL. Devuelve un **valor**:

```sql
etiqueta := CASE
                WHEN edad < 18 THEN 'menor'
                WHEN edad < 65 THEN 'adulto'
                ELSE 'mayor'
            END;
```

### Variante 2 — `CASE` como sentencia

Específica de PL/pgSQL. Ejecuta **bloques**, no devuelve valor:

```sql
CASE
    WHEN edad < 18 THEN
        RAISE NOTICE 'menor de edad';
        contador_menores := contador_menores + 1;
    WHEN edad < 65 THEN
        RAISE NOTICE 'adulto';
    ELSE
        RAISE NOTICE 'mayor';
END CASE;
```

Diferencias: la sentencia termina con `END CASE;` (no solo `END`), y cada `THEN` puede tener múltiples sentencias separadas por `;`.

In [ ]:
%%sql
DO $$
DECLARE
    region TEXT := 'CDMX';
    tipo   TEXT;
BEGIN
    -- Variante expresión: asigna un valor
    tipo := CASE region
                WHEN 'CDMX'      THEN 'metro'
                WHEN 'EdoMex'    THEN 'metro'
                WHEN 'Guerrero'  THEN 'sur'
                ELSE 'otro'
            END;
    RAISE NOTICE 'Tipo (expresión): %', tipo;
    
    -- Variante sentencia: ejecuta múltiples acciones
    CASE
        WHEN region IN ('CDMX', 'EdoMex') THEN
            RAISE NOTICE 'Es región metropolitana';
            tipo := 'metro';
        WHEN region = 'Guerrero' THEN
            RAISE NOTICE 'Es sur';
            tipo := 'sur';
        ELSE
            RAISE NOTICE 'Otra región';
    END CASE;
END
$$;

**Cuándo cada una:**

- **Expresión** — cuando solo quieres un valor. Más compacta. Se puede usar en SQL puro también.
- **Sentencia** — cuando cada rama necesita ejecutar más de una cosa. Solo en PL/pgSQL.

Si la lógica es solo asignar un valor, el `IF`/`ELSIF`/`ELSE` y el `CASE` sentencia hacen lo mismo. Convención: `CASE` cuando la decisión depende de **un solo valor de referencia** con muchos casos; `IF` cuando son **condiciones distintas**.

## `FOR` sobre rangos numéricos

Sintaxis:

```sql
FOR variable IN inicio..fin LOOP
    -- código
END LOOP;

FOR variable IN REVERSE fin..inicio LOOP        -- hacia atrás
    -- código
END LOOP;

FOR variable IN inicio..fin BY paso LOOP        -- con paso custom
    -- código
END LOOP;
```

La variable **no necesita declararse** — el `FOR` la crea automáticamente con scope del loop.

In [ ]:
%%sql
DO $$
BEGIN
    RAISE NOTICE '--- Ascendente 1..5 ---';
    FOR i IN 1..5 LOOP
        RAISE NOTICE 'i = %', i;
    END LOOP;
    
    RAISE NOTICE '--- Descendente 5..1 ---';
    FOR i IN REVERSE 5..1 LOOP
        RAISE NOTICE 'i = %', i;
    END LOOP;
    
    RAISE NOTICE '--- De 2 en 2 ---';
    FOR i IN 0..10 BY 2 LOOP
        RAISE NOTICE 'i = %', i;
    END LOOP;
END
$$;

## `FOR` sobre filas de query

**Este es el patrón más útil de PL/pgSQL.** Itera sobre el resultado de una query:

```sql
FOR row_var IN SELECT col1, col2 FROM tabla LOOP
    -- usa row_var.col1, row_var.col2
END LOOP;
```

La variable `row_var` se declara implícitamente como `RECORD` y toma una fila completa por iteración.

In [ ]:
%%sql
DO $$
DECLARE
    fila RECORD;
    total_categorias INTEGER := 0;
BEGIN
    FOR fila IN 
        SELECT category_name, COUNT(*) AS productos
          FROM northwind_dwh.dim_product
         GROUP BY category_name
         ORDER BY productos DESC
    LOOP
        RAISE NOTICE 'Categoría: % (% productos)', fila.category_name, fila.productos;
        total_categorias := total_categorias + 1;
    END LOOP;
    
    RAISE NOTICE 'Total categorías procesadas: %', total_categorias;
END
$$;

> :warning: **Anti-patrón frecuente:** usar `FOR ... IN SELECT` para hacer cálculos que se podrían expresar como `SELECT ... GROUP BY` o `UPDATE ... FROM`. PL/pgSQL es 10–100× más lento que SQL set-based.
>
> Solo usa este patrón cuando **cada fila necesita lógica realmente distinta** (e.g., enviar un email, hacer una llamada externa, manejar un error fila a fila).

## `WHILE`

Itera mientras una condición sea verdadera. Útil cuando el número de iteraciones depende de algo que cambia dentro del loop:

```sql
WHILE condicion LOOP
    -- código
END LOOP;
```

In [ ]:
%%sql
DO $$
DECLARE
    saldo NUMERIC := 1000.00;
    iter  INTEGER := 0;
BEGIN
    -- Restar 100 hasta que saldo <= 0
    WHILE saldo > 0 LOOP
        saldo := saldo - 100;
        iter  := iter + 1;
        RAISE NOTICE 'Iter %: saldo = %', iter, saldo;
    END LOOP;
    
    RAISE NOTICE 'Loop terminó después de % iteraciones', iter;
END
$$;

## Cierre

Construcciones del notebook:

| Construcción | Cuándo usarla |
|---|---|
| `IF / ELSIF / ELSE / END IF` | Condicionales con condiciones heterogéneas |
| `CASE WHEN ... THEN ... END` (expresión) | Asignar un valor según un caso |
| `CASE ... END CASE;` (sentencia) | Ejecutar bloques distintos según un caso |
| `FOR i IN 1..N LOOP` | Iteración numérica |
| `FOR row IN SELECT ... LOOP` | Iteración fila por fila de una query |
| `WHILE cond LOOP` | Iteración condicional |

**Regla operativa:** todo loop sobre filas de query es **sospechoso**. Pregúntate primero si lo puedes expresar con `UPDATE`/`INSERT SELECT`/CTE. Solo deja PL/pgSQL cuando hay lógica per-fila genuinamente irreducible.

El siguiente notebook (**03 — Procedimientos y cursores**) empaqueta esta lógica en procedimientos almacenados que se invocan con `CALL`.

---

<p align="center">
<a href="01_introduccion_y_bloques.ipynb">← Anterior: Notebook 01</a> | <a href="Readme.md">Volver al índice</a> | <a href="03_procedimientos_y_cursores.ipynb">Siguiente: Notebook 03 — Procedimientos y cursores →</a>
</p>